# 15 · A SWE-Gym-style RL environment

How to wrap coding tasks as a **Gym-style environment**
(`reset` / `step`), give an **executable reward**, run a policy, and record
**trajectories** for RL/SFT.

**Key concepts**
- **Env interface** — `reset()` returns an observation (issue + failing tests);
  `step(action)` applies the action, re-runs the suite, and returns
  `obs, reward, done, info`. This is the standard RL contract.
- **Executable reward** — reward = fraction of tests passing; `done` at 1.0. Unlike
  a static metric, it's produced by *running* code in the sandbox.
- **Action space, faithfully** — `patch` (a `git apply` diff, SWE-bench-faithful,
  microvm) vs `overwrite` (whole-file edit, portable to any backend). The notebook
  makes this explicit rather than hiding it.
- **Trajectories** — `(observation, action, reward, done)` steps are logged to JSONL
  — the data SWE-Gym exists to produce (model *training* is a separate step).

**Prerequisites:** default `microvm` + `patch` (`RAITBOX_MICROVM_IMAGE`,
`RAITBOX_EXECUTION_ROLE`) — or set `RAITBOX_SWEGYM_BACKEND=agentcore-runtime` with
the portable `overwrite` action.

In [ ]:
%pip install -q raitbox

### Configuration & task
Backend/action-format selection, per-backend `Sandbox.create` kwargs, and the buggy repo + tests the environment serves.

In [ ]:
import json
import os
import re

import boto3

from raitbox import Sandbox

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")
MODEL_ID = os.environ.get("ANTHROPIC_BEDROCK_MODEL", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")
bedrock = boto3.client("bedrock-runtime", region_name=REGION)

# --- backend + action-format selection -------------------------------------
# Faithful default: microvm + patch. Set RAITBOX_SWEGYM_BACKEND=agentcore-runtime
# (or agentcore-ci) to use the portable "overwrite" action instead.
BACKEND = os.environ.get("RAITBOX_SWEGYM_BACKEND", "microvm")
ACTION_FORMAT = os.environ.get("RAITBOX_SWEGYM_ACTION", "patch" if BACKEND == "microvm" else "overwrite")
assert not (ACTION_FORMAT == "patch" and BACKEND != "microvm"), \
    "patch/git apply needs the microvm backend; use ACTION_FORMAT=overwrite elsewhere"

# Per-backend Sandbox.create(...) kwargs.
if BACKEND == "microvm":
    SANDBOX_KWARGS = dict(
        backend="microvm",
        template=os.environ["RAITBOX_MICROVM_IMAGE"],
        execution_role_arn=os.environ["RAITBOX_EXECUTION_ROLE"],
        egress_connectors=[os.environ.get(
            "RAITBOX_EGRESS",
            f"arn:aws:lambda:{REGION}:aws:network-connector:aws-network-connector:INTERNET_EGRESS")],
        region_name=REGION, timeout=1800,
    )
elif BACKEND == "agentcore-runtime":
    SANDBOX_KWARGS = dict(backend="agentcore-runtime",
                          template=os.environ["RAITBOX_RUNTIME_ARN"], region_name=REGION)
else:  # agentcore-ci
    SANDBOX_KWARGS = dict(region_name=REGION)

print(f"backend={BACKEND}  action_format={ACTION_FORMAT}")

# --- the task (a tiny buggy repo + its test suite) --------------------------
ISSUE = ("sum_even() should return the sum of the EVEN numbers, but it currently "
         "sums the odd ones. Fix calc.py so the tests pass.")
BUGGY_SRC = "def sum_even(nums):\n    return sum(n for n in nums if n % 2 == 1)\n"
TEST_SRC = "\n".join([
    "import unittest",
    "from calc import sum_even",
    "class T(unittest.TestCase):",
    "    def test_basic(self):",
    "        self.assertEqual(sum_even([1,2,3,4]), 6)",
    "    def test_negatives(self):",
    "        self.assertEqual(sum_even([-2,-1,0,3,4]), 2)",
    "    def test_empty(self):",
    "        self.assertEqual(sum_even([]), 0)",
    "if __name__ == '__main__':",
    "    unittest.main()",
])

## The environment: `reset()` / `step()` with an executable reward

In [ ]:
class SWEEnv:
    """SWE-Gym-style env over an RaitBox sandbox, backend/action-format agnostic.

    action_format="patch"     -> action {"diff": <unified diff>}, applied with git apply (microvm)
    action_format="overwrite" -> action {"source": <full calc.py>}, written directly (any backend)
    Reward = fraction of the repo's tests passing.
    """

    REPO = "/repo"

    def __init__(self):
        self.sbx = None

    def _run_tests(self):
        res = self.sbx.commands.run(f"cd {self.REPO} && python3 -m unittest -v 2>&1 || true", timeout=120)
        out = res.stdout
        m = re.search(r"Ran (\d+) test", out)
        total = int(m.group(1)) if m else 0
        passed = len(re.findall(r"\.\.\. ok", out))
        return (passed / total if total else 0.0), out

    def _observation(self, test_output):
        return {"issue": ISSUE,
                "source": self.sbx.files.read(f"{self.REPO}/calc.py"),
                "test_output": "\n".join(test_output.splitlines()[-25:])}

    def reset(self):
        self.sbx = Sandbox.create(**SANDBOX_KWARGS)
        self.sbx.files.write(f"{self.REPO}/calc.py", BUGGY_SRC)
        self.sbx.files.write(f"{self.REPO}/test_calc.py", TEST_SRC)
        if ACTION_FORMAT == "patch":
            # NOTE: for real instances, bake git into the image instead of installing here.
            self.sbx.commands.run("dnf install -y git >/dev/null 2>&1; true")
            self.sbx.commands.run(f"cd {self.REPO} && git init -q && git add -A && "
                                  "git -c user.email=a@b.c -c user.name=x commit -qm init")
        reward, out = self._run_tests()
        return self._observation(out)

    def step(self, action):
        info = {}
        if ACTION_FORMAT == "patch":
            self.sbx.files.write(f"{self.REPO}/fix.patch", action["diff"])
            ap = self.sbx.commands.run(f"cd {self.REPO} && git apply fix.patch 2>&1 && echo APPLIED || echo FAILED")
            info["apply"] = (ap.stdout.strip().splitlines() or [""])[-1]
        else:  # overwrite
            self.sbx.files.write(f"{self.REPO}/calc.py", action["source"])
            info["apply"] = "OVERWRITE"
        reward, out = self._run_tests()
        done = reward >= 1.0
        return self._observation(out), reward, done, info

    def close(self):
        if self.sbx:
            self.sbx.kill()

## The policy: Claude (Bedrock) proposes an action matching the action format

In [ ]:
def policy(obs):
    if ACTION_FORMAT == "patch":
        rule = ("reply with ONLY a unified diff (git format, paths a/calc.py b/calc.py) "
                "that makes the tests pass")
        key = "diff"
    else:
        rule = "reply with ONLY the full corrected contents of calc.py"
        key = "source"
    system = [{"text": f"You are a software engineer fixing a bug. {rule}. No prose, no markdown fences."}]
    user = (f"Issue:\n{obs['issue']}\n\ncalc.py:\n{obs['source']}\n\n"
            f"Failing tests:\n{obs['test_output']}")
    resp = bedrock.converse(modelId=MODEL_ID, system=system,
                            messages=[{"role": "user", "content": [{"text": user}]}],
                            inferenceConfig={"maxTokens": 800, "temperature": 0})
    text = "".join(b.get("text", "") for b in resp["output"]["message"]["content"])
    if key == "diff":
        m = re.search(r"(diff --git .*|--- a/.*)", text, re.DOTALL)
        return {"diff": (m.group(0) if m else text).strip() + "\n"}
    fenced = re.findall(r"```(?:python)?\n(.*?)```", text, re.DOTALL)
    return {"source": (fenced[0] if fenced else text).strip() + "\n"}

## Run one episode and collect the trajectory

In [ ]:
def run_episode(max_steps=4):
    env = SWEEnv()
    trajectory = []
    try:
        obs = env.reset()
        for t in range(1, max_steps + 1):
            action = policy(obs)
            next_obs, reward, done, info = env.step(action)
            trajectory.append({"step": t, "observation": obs, "action": action,
                               "reward": reward, "done": done, "info": info})
            print(f"step {t}: reward {reward:.2f} done={done} apply={info.get('apply')}")
            obs = next_obs
            if done:
                break
    finally:
        env.close()
    return trajectory


trajectory = run_episode()
resolved = bool(trajectory) and trajectory[-1]["done"]
print("\nRESOLVED" if resolved else "\nNOT RESOLVED", "in", len(trajectory), "step(s)")

## Persist the trajectory as JSONL (SFT/RL-ready), then reload

In [ ]:
path = "swe_gym_trajectory.jsonl"
with open(path, "w") as f:
    for record in trajectory:
        f.write(json.dumps(record) + "\n")
print("wrote", path, "-", len(trajectory), "steps")

for line in open(path):
    r = json.loads(line)
    print(f"step {r['step']}: reward={r['reward']:.2f} action={list(r['action'])[0]} done={r['done']}")

## (Optional) Parallel rollout farm — a vectorized env

Run several episodes concurrently, each in its **own** sandbox, and aggregate the
resolve-rate. Spins up N sandboxes at once (bills N sessions); `RAITBOX_MAX_SESSIONS`
caps concurrency.

In [ ]:
import asyncio

N_EPISODES = int(os.environ.get("SWE_GYM_EPISODES", "2"))

async def rollout_farm(n):
    trajs = await asyncio.gather(*(asyncio.to_thread(run_episode) for _ in range(n)))
    resolved = sum(1 for tr in trajs if tr and tr[-1]["done"])
    return trajs, resolved

trajs, resolved = await rollout_farm(N_EPISODES)
print(f"resolve-rate: {resolved}/{N_EPISODES}")

## Recap & where to go next
You built a reusable RL environment with an executable reward and emitted a
trajectory dataset — the bridge from evaluation to training data.

**Related:** `14` (the patch loop it generalizes) · `07` (sampling) ·
`08` (synthesize the tasks such an env trains on).